# Phase 4 — LLM-planned arm (smoke test + N-run batch)

Answers the same research question as `02_rule_based_arm.ipynb`, but by
handing a natural-language version of it to `s3geo.query()` and letting
the planner choose the operations, parameters and CRS itself — no
operation list, no threshold, no CRS is given. Method:
`paper/PLAN.md` "Arm 2".

**⚠️ Not yet executed — and, unlike the first two notebooks, this one
genuinely can't be smoke-tested without a live LLM call.** Every mechanic
below (exception types, entity-ref resolution, the `default=str` used to
convert layers, `temperature=0.1` as the real default) was verified
against 0.3.0 source (`s3geo/__init__.py`,
`orchestrator/planning/llm_spec_generator.py`,
`orchestrator/planning/planner.py`) — but the actual plan an LLM produces
for our exact query is unknown until it runs for real.

**This notebook deliberately stops after the smoke test (§4).** The
N=20 batch (§5) only *runs and saves raw output* — it does not try to
parse "which mahalle are underserved" out of the result, because that
depends on property names and an output shape only a real LLM plan can
reveal. Per `STUDY_LOG.md` "Smoke-test before any N-run batch": run §1-§4,
read the printed operation sequence, and only then decide whether to run
§5 — do not run the whole notebook blind.


## 1. Setup

`OpenAICompatibleLLMClient()` (`orchestrator/planning/llm_spec_generator.py`)
reads `LLM_API_KEY` / `AVALAI_API_KEY` / `OPENAI_API_KEY`, `LLM_BASE_URL`
(default `https://api.avalai.ir/v1`), and `LLM_MODEL` (default
`gpt-4o-mini`) straight from `os.environ` — nothing in `s3geo.query()`
loads `.env` for you, so this notebook does it explicitly. Temperature is
**not** a parameter of `s3geo.query()` at all — `LLMQuerySpecGenerator`
defaults to `temperature=0.1` and `s3geo.query()` never overrides it, so
every call below is already at the temperature `paper/PLAN.md` specifies,
with no extra code needed.


In [12]:
import json
import time
from dataclasses import is_dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path

import geopandas as gpd
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env")

import os
import s3geo
from orchestrator.planning.llm_spec_generator import LLMSpecGenerationError
from orchestrator.planning.planner import PlanningError

RAW = Path("../data/raw")
RESULTS = Path("../results")
LLM_RUNS = RESULTS / "llm_runs"
LLM_RUNS.mkdir(parents=True, exist_ok=True)

# Which smart-spatial-system pin this run is on - saved into every record,
# so a batch's results say what produced them (earlier batches had to be
# matched to a pin by file timestamps).
from importlib.metadata import version as _pkg_version
SSS_VERSION = _pkg_version("smart-spatial-system")
print(f"smart-spatial-system installed: {SSS_VERSION}")


def archive_previous_batch(batch_dir: Path) -> None:
    """
    Copy an existing batch (run_*.json + manifest.csv) into
    results/archive/<dir>_<manifest timestamp>/ before this notebook
    overwrites it. Copy, never move; skipped if already archived.
    """
    import shutil
    manifest_path = batch_dir / "manifest.csv"
    if not manifest_path.exists():
        return
    stamp = datetime.fromtimestamp(manifest_path.stat().st_mtime, tz=timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    dest = RESULTS / "archive" / f"{batch_dir.name}_{stamp}"
    if dest.exists():
        print(f"already archived: {dest}")
        return
    dest.mkdir(parents=True)
    for f in sorted(batch_dir.glob("run_*.json")) + [manifest_path]:
        shutil.copy2(f, dest / f.name)
    print(f"archived previous batch {batch_dir} -> {dest}")


archive_previous_batch(LLM_RUNS)
archive_previous_batch(RESULTS / "llm_runs_mitigated")

N_RUNS = 20
SLEEP_BETWEEN_CALLS_S = 1.0  # be polite to a shared endpoint

_api_key_present = bool(
    os.environ.get("LLM_API_KEY")
    or os.environ.get("AVALAI_API_KEY")
    or os.environ.get("OPENAI_API_KEY")
)
_base_url = os.environ.get("LLM_BASE_URL", "https://api.avalai.ir/v1 (default)")
_model = os.environ.get("LLM_MODEL", "gpt-4o-mini (default)")

print(f"LLM API key present: {_api_key_present}")
print(f"LLM_BASE_URL: {_base_url}")
print(f"LLM_MODEL: {_model}")
print(f"temperature: 0.1 (LLMQuerySpecGenerator's own default, not overridden)")

if not _api_key_present:
    raise RuntimeError(
        "No LLM_API_KEY / AVALAI_API_KEY / OPENAI_API_KEY found in the environment "
        "after loading ../.env. Fix .env before continuing -- per STUDY_LOG.md, don't "
        "let this phase silently run with no key and just fail cryptically per call."
    )


smart-spatial-system installed: 0.5.6
archived previous batch ../results/llm_runs -> ../results/archive/llm_runs_20260923T194447Z
archived previous batch ../results/llm_runs_mitigated -> ../results/archive/llm_runs_mitigated_20260923T195003Z
LLM API key present: True
LLM_BASE_URL: https://api.avalai.ir/v1
LLM_MODEL: gpt-4o-mini
temperature: 0.1 (LLMQuerySpecGenerator's own default, not overridden)


## 2. Load layers (EPSG:4326 — deliberately *not* reprojected)

Unlike `02_rule_based_arm.ipynb`, this arm must see the data in its
**original, unprojected** form. Handing it already-`EPSG:32635` data
would silently answer the CRS question for it — and per `STUDY_LOG.md`
"The LLM arm chooses its own parameters", whether and how to reproject is
exactly what's being measured.

Loads `data/raw/` directly, same defensive clean-check as
`02_rule_based_arm.ipynb` §1 (that notebook's real run found 0 null/
invalid geometries here — re-asserted, not trusted blindly).

**Not affected by `bugs/001`**: `s3geo.query()`'s own layer conversion
(`s3geo/__init__.py::_to_geojson_dict`) calls `layer.to_json(default=str)`
— unlike `VectorOut.from_geopandas()`, it already passes `default=str`,
so the `datetime64` `check_date`/`start_date` columns that crashed Arm 1's
first run don't crash this one. Worth folding back into `bugs/001`'s
proposed fix as a precedent: the framework's own code already handles
this correctly in one place.


In [13]:
hospitals_raw = gpd.read_file(RAW / "hospitals.geojson")
mahalle_raw = gpd.read_file(RAW / "mahalle_boundaries.geojson")

assert hospitals_raw.crs is not None and hospitals_raw.crs.to_string().upper() == "EPSG:4326"
assert mahalle_raw.crs is not None and mahalle_raw.crs.to_string().upper() == "EPSG:4326"

null_hosp = hospitals_raw.geometry.isna().sum()
null_mah = mahalle_raw.geometry.isna().sum()
invalid_mah = (~mahalle_raw.geometry.is_valid).sum()
if null_hosp or null_mah or invalid_mah:
    raise ValueError(
        "data/raw/ is not as clean as 01_data_and_problem.ipynb found it -- "
        "stop and inspect before spending LLM calls on it."
    )

print(f"hospitals_raw: {len(hospitals_raw)} features, CRS={hospitals_raw.crs}")
print(f"mahalle_raw:   {len(mahalle_raw)} features, CRS={mahalle_raw.crs}")


hospitals_raw: 1020 features, CRS=EPSG:4326
mahalle_raw:   964 features, CRS=EPSG:4326


## 3. The raw query — identical across every run, defined once

States the question, the two layers (named to match the `layers=` dict
keys below **exactly** — see the note under §4 on why that match matters),
and the city. Deliberately omits the distance threshold, the target CRS,
the centroid-vs-boundary choice, and the operation list — per `STUDY_LOG.md`
"The LLM arm chooses its own parameters", choosing those is the thing
being measured. States the input CRS as a fact about the data (an LLM
can't plan a CRS transform sensibly with no idea what CRS it's starting
from) without telling it *what to do about it* or *what to change it to*.


In [14]:
RAW_QUERY = (
    "Layers: 'hospitals' is a point layer of hospitals and clinics in "
    "Istanbul, Turkey, in EPSG:4326. 'mahalle' is a polygon layer of "
    "Istanbul neighbourhood (mahalle) boundaries, also in EPSG:4326. "
    "For each mahalle, determine how far it is from the nearest hospital "
    "or clinic, and identify which mahalle are underserved because their "
    "nearest facility is too far away."
)
print(RAW_QUERY)


Layers: 'hospitals' is a point layer of hospitals and clinics in Istanbul, Turkey, in EPSG:4326. 'mahalle' is a polygon layer of Istanbul neighbourhood (mahalle) boundaries, also in EPSG:4326. For each mahalle, determine how far it is from the nearest hospital or clinic, and identify which mahalle are underserved because their nearest facility is too far away.


## 3b. What the framework will tell the planner about this data (no LLM call)

From `smart-spatial-system` 0.5.5 on (`enhancements/002` item 1),
`s3geo.query()` computes the combined EPSG:4326 extent of `layers` and a
projected CRS for metric work on it (UTM zone of the centroid, only when
one zone is accurate within 1% across the whole extent), and states both
to the planner as an "Input data facts" section of the system prompt.
This cell calls the same function `query()` calls, on the same layers,
so the facts are visible before any API call is spent.

It asserts the suggestion is `EPSG:32635` - Arm 1's own CRS. Checked
before this cell was written, by running the pyproj-free parts of
`input_data_extent.py` on this repo's real `data/raw/` files: combined
bbox lon 27.9708-29.9588, lat 40.8027-41.5833, both edges in UTM zone 35,
max scale error about 0.04%. If the assert fails, stop and find out why
before spending LLM calls. `None` means pyproj is missing or a layer's CRS
is unknown - the planner then gets no data facts at all, and the 0.5.4
placeholder failure mode is expected back (now caught at generation).

In [15]:
from orchestrator.planning.input_data_extent import (
    derive_input_data_extent,
    render_input_data_facts,
)

input_extent = derive_input_data_extent({"hospitals": hospitals_raw, "mahalle": mahalle_raw})

if input_extent is None:
    raise RuntimeError(
        "derive_input_data_extent returned None - pyproj missing or a layer has an unknown CRS. "
        "The planner would get no 'Input data facts' section; fix this before running the batch."
    )

print(render_input_data_facts(input_extent))
print(f"\nmax scale error of the suggested CRS across the extent: {input_extent.max_scale_error:.4%}")
assert input_extent.suggested_crs == "EPSG:32635", (
    f"framework computed {input_extent.suggested_crs!r}, expected EPSG:32635 (Arm 1's CRS) - investigate before running"
)

Input data facts (computed from THIS query's own input layers - these are measured facts about the data, not an example):
- Input layers and the CRS each is stored in: hospitals (EPSG:4326), mahalle (EPSG:4326)
- Combined extent in EPSG:4326 (lon/lat): min_lon=27.970848, min_lat=40.802704, max_lon=29.958805, max_lat=41.583302; centroid lon=28.964826, lat=41.193003.
- Suitable projected CRS for metric work on this data: EPSG:32635 (WGS 84 / UTM zone 35N). Use exactly this value as target_crs for every crs_transform that prepares data for a distance, nearest-neighbor, buffer or area operation, and as source_crs/target_crs on the distance operation itself - unless the user explicitly asks for a different CRS.

max scale error of the suggested CRS across the extent: 0.0367%


## 4. Smoke test — ONE call

`STUDY_LOG.md` "Smoke-test before any N-run batch": one call, printed plan,
eyes on the operation sequence and per-layer CRS handling, before any
batch. Vienna burned three full N=20 batches on mistakes this would have
caught.

**Why the layer names must match exactly**: verified from
`orchestrator/planning/planner.py` — `PlannerConfig.allow_implicit_entities`
defaults to `True`, so *any* input-ref string the LLM's plan uses that
isn't another operation's output becomes `$inputs.<that string>` and is
looked up directly in the `layers=` dict passed to `s3geo.query()`
(`orchestrator/planning/dag_executor.py::_resolve_ref`). There is no
schema/binding step in between — if the LLM's plan calls the hospitals
layer anything other than exactly `"hospitals"`, the run fails at
execution with an unresolvable reference, not a wrong answer. That's why
§3's query states the layer names as quoted literals matching the dict
keys below.

`_output_for_json()` below exists because `result.output` can legitimately
be several different shapes depending on which operation the LLM's plan
ends on — a `VectorOut` (has `.features`), a `ReportOut` (a dataclass,
from `build_report`), or a plain dict/list — and this notebook has no way
to know which until a real plan comes back.


In [16]:
def _output_for_json(output):
    """Best-effort JSON-safe view of result.output, whatever shape it is."""
    if is_dataclass(output) and not isinstance(output, type):
        return asdict(output)
    if hasattr(output, "features"):
        return {"features": output.features, "metadata": getattr(output, "metadata", {})}
    if isinstance(output, (dict, list, str, int, float, bool)) or output is None:
        return output
    return repr(output)


def _attempts_for_json(attempts):
    """[SpecGenerationAttempt, ...] -> JSON-safe list, or None if empty/absent."""
    if not attempts:
        return None
    return [asdict(a) for a in attempts]


def run_once(raw_query: str, *, layers: dict, system_hints: str | None = None) -> dict:
    """
    One s3geo.query() call -> a JSON-serializable record.
    Success or failure, every field this notebook can observe is captured --
    a failure is data, not something to retry away (STUDY_LOG.md).

    0.5.6 update: s3geo.query() now defaults to max_repair_attempts=1 --
    a plan rejected at generation gets one re-prompt before failing for
    real. That means "success" alone no longer says whether the accepted
    plan was the LLM's first try or a repaired one, which matters for
    this study (enhancements/003 exists because of exactly that gap).
    S3GeoResult.attempt_count/.repaired/.generation_attempts, and the
    matching attributes on LLMSpecGenerationError/S3GeoExecutionError,
    make that visible now. getattr(..., None) guards throughout so this
    still runs cleanly (fields just stay None) if ever pointed at an
    older pin.
    """
    started = time.monotonic()
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "raw_query": raw_query,
        "model": _model,
        "success": False,
        "error_stage": None,
        "error": None,
        "goal": None,
        "operations": None,
        "output": None,
        "system_hints_used": bool(system_hints),
        "smart_spatial_system_version": SSS_VERSION,
        "input_data_extent": None,
        # 0.5.6+ repair visibility -- see docstring above.
        "attempt_count": None,
        "repaired": None,
        "generation_attempts": None,
        "query_spec": None,
        "plan": None,
    }
    try:
        result = s3geo.query(raw_query, layers=layers, system_hints=system_hints or "")
        record["success"] = True
        record["goal"] = result.goal
        record["operations"] = result.operations
        record["output"] = _output_for_json(result.output)
        # 0.5.5+: what s3geo.query() measured from the layers and told the
        # planner (extent + suggested CRS). Absent on older pins.
        _extent = getattr(result, "input_data_extent", None)
        record["input_data_extent"] = asdict(_extent) if _extent is not None else None
        # 0.5.6+: was the accepted plan the first one, or did it take a
        # generation-time rejection + repair to get here?
        _attempts = getattr(result, "generation_attempts", None)
        if _attempts is not None:
            record["attempt_count"] = result.attempt_count
            record["repaired"] = result.repaired
            record["generation_attempts"] = _attempts_for_json(_attempts)
        record["query_spec"] = getattr(result, "query_spec", None)
        record["plan"] = getattr(result, "plan", None)
    except LLMSpecGenerationError as exc:
        record["error_stage"] = "generation"  # LLM's plan failed generation-time validation
        record["error"] = str(exc)
        # 0.5.6+: even a final rejection (repair exhausted) carries the
        # attempt trail and the last rejected plan on the exception itself.
        # Note the attribute is `.attempts` here, not `.generation_attempts`
        # (S3GeoExecutionError below uses the other name) -- checked
        # against the installed 0.5.6 source, not assumed.
        _exc_attempts = getattr(exc, "attempts", None)
        if _exc_attempts is not None:
            record["generation_attempts"] = _attempts_for_json(_exc_attempts)
            record["attempt_count"] = len(_exc_attempts)
            record["repaired"] = False  # repair was attempted and still failed
        record["plan"] = getattr(exc, "plan", None)
    except PlanningError as exc:
        record["error_stage"] = "planning"  # QuerySpec -> DagPlan failed
        record["error"] = str(exc)
    except RuntimeError as exc:
        record["error_stage"] = "execution"  # DagPlan built but failed running
        record["error"] = str(exc)
        # 0.5.6+: S3GeoExecutionError (a RuntimeError subclass, so it's
        # already caught here) carries query_spec/plan/generation_attempts
        # for the plan that was executing when the DAG failed.
        _exc_attempts = getattr(exc, "generation_attempts", None)
        if _exc_attempts is not None:
            record["generation_attempts"] = _attempts_for_json(_exc_attempts)
            record["attempt_count"] = len(_exc_attempts)
            record["repaired"] = len(_exc_attempts) > 1
        record["query_spec"] = getattr(exc, "query_spec", None)
        record["plan"] = getattr(exc, "plan", None)
    record["latency_s"] = round(time.monotonic() - started, 3)
    return record


layers = {"hospitals": hospitals_raw, "mahalle": mahalle_raw}

smoke = run_once(RAW_QUERY, layers=layers)

print(f"success: {smoke['success']}  latency: {smoke['latency_s']}s")
if not smoke["success"]:
    print(f"FAILED at stage={smoke['error_stage']}: {smoke['error']}")
    if smoke["attempt_count"]:
        print(f"  ({smoke['attempt_count']} generation attempt(s) made before failing)")
else:
    print(f"goal: {smoke['goal']}")
    print(f"operations ({len(smoke['operations'])}): {smoke['operations']}")
    crs_transform_calls = smoke['operations'].count('crs_transform')
    print(f"crs_transform calls: {crs_transform_calls} "
          f"{'(expected >= 2, one per layer, for a real metric distance)' if crs_transform_calls < 2 else '(OK)'}")
    if smoke["attempt_count"] is not None:
        print(f"generation attempts: {smoke['attempt_count']}  repaired: {smoke['repaired']}")
    out = smoke["output"]
    if isinstance(out, dict) and "features" in out:
        print(f"output: VectorOut-shaped, {len(out['features'])} features")
        if out["features"]:
            print("sample feature properties:", json.dumps(out["features"][0].get("properties", {}), indent=2, ensure_ascii=False))
    else:
        print(f"output type: {type(smoke['output']).__name__}")
        print(json.dumps(out, indent=2, ensure_ascii=False)[:3000])

(LLM_RUNS / "run_smoke.json").write_text(json.dumps(smoke, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nwrote {LLM_RUNS / 'run_smoke.json'}")


success: True  latency: 14.773s
goal: Determine distance to nearest hospital for each mahalle
operations (4): ['crs_transform', 'crs_transform', 'spatial_nearest', 'filter_attribute']
crs_transform calls: 2 (OK)
generation attempts: 1  repaired: False
output: VectorOut-shaped, 166 features
sample feature properties: {
  "osm_id": 9064926,
  "osm_type": "relation",
  "admin_level": "8",
  "alt_name": null,
  "boundary": "administrative",
  "description": null,
  "name": "Yazımanayır Mahallesi",
  "name:ar": null,
  "name:el": null,
  "name:tr": null,
  "name:uz": null,
  "network": null,
  "note": null,
  "place": null,
  "population": null,
  "population:date": null,
  "postal_code": null,
  "short_name": null,
  "source:population": null,
  "type": "boundary",
  "wikidata": "Q97228716",
  "wikimedia_commons": null,
  "wikipedia": null,
  "distance_to_hospital": 1010.009162,
  "_neighbor_rank": 1,
  "_source_index": 238,
  "_target_index": 814,
  "_nearest_status": "matched",
  "_neare

---
## STOP

Read the printed `operations` list and the sample output above before
running §5.

- If it failed: read `error_stage`/`error` in `run_smoke.json`. A
  `generation` failure means the LLM's own plan tripped one of
  `llm_spec_generator.py`'s validators (e.g. the CRS-symmetry check, or
  the `filter_points_in_polygon`-without-identity check) — that's the
  framework catching a bad plan before it runs, working as intended, not
  itself a bug. A `planning` or `execution` failure is worth a closer look
  before batching 20 more calls at it.
- If it succeeded but `crs_transform` appears 0 or 1 times: the plan
  likely computed a meaningless cross-CRS distance (or skipped distance
  entirely) — per `paper/PLAN.md` this is itself a real, reportable
  result about how the LLM arm behaves, not something to fix by rerunning
  until it looks better.
- If it succeeded cleanly: check the sample feature's properties for
  something that looks like a distance value and an underserved flag —
  whatever the LLM actually named them. That real property name is what
  the next notebook's extraction logic needs to be built around, not a
  guess made ahead of time.

**Do not run §5 automatically — come back to this notebook once the smoke
test's real output has been reviewed.**

---


## 5. N = 20 runs

Every run — success or failure — saved to `results/llm_runs/run_{i:02d}.json`
verbatim (same `run_once()` record shape as the smoke test), indexed in
`manifest.csv`. This cell **only executes and persists** — it does not
attempt to extract "which mahalle are underserved" from `output`, since
that depends on property names only a real run reveals (see §4's STOP).
That extraction is `04_comparison_metric.ipynb`'s job, informed by what
the smoke test (and this batch) actually produced.


In [17]:
manifest_rows = []

for i in range(N_RUNS):
    record = run_once(RAW_QUERY, layers=layers)
    run_path = LLM_RUNS / f"run_{i:02d}.json"
    run_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")

    n_ops = len(record["operations"]) if record["operations"] else 0
    n_crs = record["operations"].count("crs_transform") if record["operations"] else 0
    manifest_rows.append({
        "run_index": i,
        "success": record["success"],
        "error_stage": record["error_stage"],
        "goal": record["goal"],
        "n_operations": n_ops,
        "n_crs_transform_calls": n_crs,
        "latency_s": record["latency_s"],
        # 0.5.6+: None on an older pin, otherwise whether the accepted (or
        # finally-rejected) plan took a repair to get there.
        "attempt_count": record["attempt_count"],
        "repaired": record["repaired"],
    })
    status = "OK" if record["success"] else f"FAILED ({record['error_stage']})"
    repair_note = ""
    if record["attempt_count"]:
        repair_note = f"  attempts={record['attempt_count']} repaired={record['repaired']}"
    print(f"run {i:02d}: {status}  latency={record['latency_s']}s  ops={n_ops}{repair_note}")

    if i < N_RUNS - 1:
        time.sleep(SLEEP_BETWEEN_CALLS_S)

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(LLM_RUNS / "manifest.csv", index=False)

n_success = int(manifest["success"].sum())
print(f"\n{n_success} / {N_RUNS} runs succeeded")
print(manifest["error_stage"].value_counts(dropna=False))
if manifest["attempt_count"].notna().any():
    n_repaired = int(manifest["repaired"].fillna(False).sum())
    print(f"{n_repaired} / {N_RUNS} needed a repair attempt to reach their final state "
          f"(0.5.6+ max_repair_attempts=1)")
print(f"wrote {LLM_RUNS / 'manifest.csv'}")


run 00: OK  latency=19.779s  ops=4  attempts=1 repaired=False
run 01: OK  latency=15.002s  ops=4  attempts=1 repaired=False
run 02: OK  latency=16.198s  ops=4  attempts=1 repaired=False
run 03: OK  latency=14.156s  ops=4  attempts=1 repaired=False
run 04: OK  latency=13.594s  ops=4  attempts=1 repaired=False
run 05: OK  latency=14.521s  ops=4  attempts=1 repaired=False
run 06: OK  latency=15.53s  ops=4  attempts=1 repaired=False
run 07: OK  latency=13.125s  ops=4  attempts=1 repaired=False
run 08: OK  latency=13.439s  ops=4  attempts=1 repaired=False
run 09: OK  latency=13.824s  ops=4  attempts=1 repaired=False
run 10: OK  latency=15.942s  ops=4  attempts=1 repaired=False
run 11: OK  latency=14.121s  ops=4  attempts=1 repaired=False
run 12: OK  latency=13.798s  ops=4  attempts=1 repaired=False
run 13: OK  latency=15.951s  ops=4  attempts=1 repaired=False
run 14: OK  latency=14.173s  ops=4  attempts=1 repaired=False
run 15: OK  latency=15.167s  ops=4  attempts=1 repaired=False
run 16: O

## 6. Summary


In [18]:
summary = {
    "n_runs": N_RUNS,
    "n_success": int(manifest["success"].sum()),
    "success_rate": round(float(manifest["success"].mean()), 3),
    "error_stage_counts": manifest["error_stage"].value_counts(dropna=False).to_dict(),
    "mean_latency_s": round(float(manifest["latency_s"].mean()), 3),
    "distinct_goals": sorted(manifest["goal"].dropna().unique().tolist()),
    "successful_runs_with_0_or_1_crs_transform_calls": int(
        (manifest["success"] & (manifest["n_crs_transform_calls"] < 2)).sum()
    ),
    # 0.5.6+: None/omitted if this manifest predates the repair-tracking
    # columns (an older pin, or a batch run before this notebook update).
    "n_repaired": (
        int(manifest["repaired"].fillna(False).sum())
        if manifest["attempt_count"].notna().any() else None
    ),
    "model": _model,
    "temperature": 0.1,
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "n_runs": 20,
  "n_success": 20,
  "success_rate": 1.0,
  "error_stage_counts": {
    "null": 20
  },
  "mean_latency_s": 15.478,
  "distinct_goals": [
    "Determine distance from mahalle to nearest hospital and identify underserved mahalle",
    "Determine distance to nearest hospital for each mahalle",
    "Determine distance to nearest hospital for each mahalle and identify underserved areas"
  ],
  "successful_runs_with_0_or_1_crs_transform_calls": 0,
  "n_repaired": 0,
  "model": "gpt-4o-mini",
  "temperature": 0.1
}


## 7. Mitigated comparison batch (`system_hints` safety net) — NOT a replacement for §5

§5 above is the canonical, unmitigated 0.5.3 batch and stays as-is —
18/20 structural success, but **0/20 fully valid answers**, for two
reasons neither bug fix touches (`STUDY_LOG.md`/`paper/PLAN.md` "Findings",
0.5.3 batch entry, has the full analysis):

1. Every successful run reprojects to `EPSG:31256` ("MGI / Austria GK
   East") — geographically meaningless for Istanbul. Not caught by
   `_validate_distance_op_crs_symmetry`, which only checks that both
   layers share the *same* CRS, never that the CRS fits the data.
2. 7-8 of 18 runs set `nearest_neighbor.max_distance=5000.0`, which
   silently zeroes the downstream `filter_attribute` result every time
   — the plan is internally consistent per-operation but self-defeating
   as a whole.

Both are **LLM planning-choice problems, not framework bugs** — every
parameter behaves exactly as `smart_spatial_system` documents it. Per
`STUDY_LOG.md`'s own rule, a real bug gets a `bugs/` report and a stop; a
planning-choice gap like this instead gets (a) a paper Finding (done)
and, optionally, (b) a **named, documented local mitigation** — never a
silent workaround, and never an edit to the pinned package.

This section is that documented mitigation: `s3geo.query()` accepts a
free-text `system_hints` string appended verbatim to the LLM's system
prompt (`orchestrator/planning/llm_spec_generator.py:1289-1297`, cloned
`v0.5.3` source) — the framework's own
`examples/urmia_real_estate_ranking.py` uses exactly this mechanism to
supply domain facts a generic prompt wouldn't otherwise carry. `SYSTEM_HINTS`
below states two general facts a competent human analyst would already
know before touching this data — the region's correct projected CRS, and
a general DAG-composition rule about not letting an upstream distance
cap silently defeat a downstream distance filter — **not** the study's
actual threshold or its answer. That distinction is what keeps this a
mitigation of a framework/prompting gap rather than an answer handed to
the model.

**Caveat, worth remembering (the same one `bugs/004`'s "Local
mitigation" section raised for a related case): a hand-written hint like
this is only as good as it is current.** If a future `smart_spatial_system`
release changes how `max_distance`/`where` interact, or adds the upstream
validators requested in the enhancement prompt below, this hint could go
stale or become redundant without anything here flagging that — re-check
it against the changelog on every future pin bump, the same discipline
already used for every `bugs/` fix in this study.

**This batch is a comparison, not a replacement.** Both `results/llm_runs/`
(unmitigated, canonical) and `results/llm_runs_mitigated/` (this section)
are kept and reported side by side in the paper — the gap between them
*is* the finding: how much of Arm 2's failure was addressable with a
few sentences of domain grounding versus how much needs an actual
framework change.

**Update for 0.5.5 and later:** the framework now covers both points in
`SYSTEM_HINTS` itself. It states the CRS computed from the data (§3b), and
`_validate_max_distance_filter_composition()` rejects a provably empty
`max_distance`+`where` plan at generation. `SYSTEM_HINTS` is kept
unchanged so this batch stays comparable with the 0.5.4 one. From 0.5.5 on,
the gap between §5 and §7 measures whether the local hints still add
anything beyond what the framework does.

In [19]:
SYSTEM_HINTS = (
    "Domain context for this query, not part of the question being asked:\n"
    "- The data (both layers) covers Istanbul, Turkey, roughly 41.0N 29.0E. "
    "For metric distance calculations here, reproject to EPSG:32635 "
    "(WGS 84 / UTM zone 35N) -- the correct UTM zone for this longitude -- "
    "rather than an arbitrary projected CRS unrelated to this region.\n"
    "- When a plan computes nearest-neighbor distances that a later step "
    "will filter by a distance threshold, do not cap the neighbor search's "
    "own max_distance below (or anywhere near) that threshold -- doing so "
    "silently discards candidates before the filter step ever runs, which "
    "can make the filter's threshold impossible to satisfy even though "
    "each operation individually behaves exactly as documented."
)
print(SYSTEM_HINTS)

Domain context for this query, not part of the question being asked:
- The data (both layers) covers Istanbul, Turkey, roughly 41.0N 29.0E. For metric distance calculations here, reproject to EPSG:32635 (WGS 84 / UTM zone 35N) -- the correct UTM zone for this longitude -- rather than an arbitrary projected CRS unrelated to this region.
- When a plan computes nearest-neighbor distances that a later step will filter by a distance threshold, do not cap the neighbor search's own max_distance below (or anywhere near) that threshold -- doing so silently discards candidates before the filter step ever runs, which can make the filter's threshold impossible to satisfy even though each operation individually behaves exactly as documented.


In [20]:
N_RUNS_MITIGATED = 20
LLM_RUNS_MITIGATED = RESULTS / "llm_runs_mitigated"
LLM_RUNS_MITIGATED.mkdir(parents=True, exist_ok=True)

mitigated_smoke = run_once(RAW_QUERY, layers=layers, system_hints=SYSTEM_HINTS)
(LLM_RUNS_MITIGATED / "run_smoke.json").write_text(
    json.dumps(mitigated_smoke, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"mitigated smoke: success={mitigated_smoke['success']}  latency={mitigated_smoke['latency_s']}s")
if mitigated_smoke["success"]:
    print(f"operations: {mitigated_smoke['operations']}")
else:
    print(f"FAILED at stage={mitigated_smoke['error_stage']}: {mitigated_smoke['error']}")

mitigated smoke: success=True  latency=14.695s
operations: ['crs_transform', 'crs_transform', 'spatial_nearest', 'filter_attribute']


### STOP — same discipline as §4

Read the smoke result above before spending 20 more calls. If
`system_hints` broke something the unmitigated smoke test didn't (a new
`error_stage`, an unrelated operation change), stop and investigate
before running the loop below — a mitigation that introduces its own new
failure mode needs to be documented and possibly rewritten, not run past.

In [21]:
manifest_mitigated_rows = []

for i in range(N_RUNS_MITIGATED):
    record = run_once(RAW_QUERY, layers=layers, system_hints=SYSTEM_HINTS)
    run_path = LLM_RUNS_MITIGATED / f"run_{i:02d}.json"
    run_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")

    n_ops = len(record["operations"]) if record["operations"] else 0
    n_crs = record["operations"].count("crs_transform") if record["operations"] else 0
    manifest_mitigated_rows.append({
        "run_index": i,
        "success": record["success"],
        "error_stage": record["error_stage"],
        "goal": record["goal"],
        "n_operations": n_ops,
        "n_crs_transform_calls": n_crs,
        "latency_s": record["latency_s"],
        "attempt_count": record["attempt_count"],
        "repaired": record["repaired"],
    })
    status = "OK" if record["success"] else f"FAILED ({record['error_stage']})"
    repair_note = ""
    if record["attempt_count"]:
        repair_note = f"  attempts={record['attempt_count']} repaired={record['repaired']}"
    print(f"run {i:02d}: {status}  latency={record['latency_s']}s  ops={n_ops}{repair_note}")

    if i < N_RUNS_MITIGATED - 1:
        time.sleep(SLEEP_BETWEEN_CALLS_S)

manifest_mitigated = pd.DataFrame(manifest_mitigated_rows)
manifest_mitigated.to_csv(LLM_RUNS_MITIGATED / "manifest.csv", index=False)

n_success_mitigated = int(manifest_mitigated["success"].sum())
print(f"\n{n_success_mitigated} / {N_RUNS_MITIGATED} mitigated runs succeeded")
print(manifest_mitigated["error_stage"].value_counts(dropna=False))
if manifest_mitigated["attempt_count"].notna().any():
    n_repaired_mitigated = int(manifest_mitigated["repaired"].fillna(False).sum())
    print(f"{n_repaired_mitigated} / {N_RUNS_MITIGATED} needed a repair attempt to reach "
          f"their final state (0.5.6+ max_repair_attempts=1)")
print(f"wrote {LLM_RUNS_MITIGATED / 'manifest.csv'}")


run 00: OK  latency=14.574s  ops=4  attempts=1 repaired=False
run 01: OK  latency=13.939s  ops=4  attempts=1 repaired=False
run 02: OK  latency=14.368s  ops=4  attempts=1 repaired=False
run 03: OK  latency=14.039s  ops=4  attempts=1 repaired=False
run 04: OK  latency=14.079s  ops=4  attempts=1 repaired=False
run 05: OK  latency=13.96s  ops=4  attempts=1 repaired=False
run 06: OK  latency=14.752s  ops=4  attempts=1 repaired=False
run 07: OK  latency=13.403s  ops=4  attempts=1 repaired=False
run 08: OK  latency=14.303s  ops=4  attempts=1 repaired=False
run 09: OK  latency=18.303s  ops=4  attempts=1 repaired=False
run 10: OK  latency=14.822s  ops=4  attempts=1 repaired=False
run 11: OK  latency=14.842s  ops=4  attempts=1 repaired=False
run 12: OK  latency=14.304s  ops=4  attempts=1 repaired=False
run 13: OK  latency=14.365s  ops=4  attempts=1 repaired=False
run 14: FAILED (generation)  latency=1.075s  ops=0
run 15: OK  latency=13.952s  ops=4  attempts=1 repaired=False
run 16: OK  latency=

## 8. Mitigated-vs-unmitigated comparison

Does NOT replace §6's summary of the canonical batch -- this reads both manifests and reports the gap side by side, plus how many mitigated runs actually landed on EPSG:32635 and a non-self-defeating max_distance, which `manifest.csv` alone can't show (that requires reading each run's own `output.metadata`, same as the analysis already done for §5's batch).

In [22]:
def _find_key_deep(obj, key):
    if isinstance(obj, dict):
        if key in obj:
            return obj[key]
        for v in obj.values():
            r = _find_key_deep(v, key)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = _find_key_deep(v, key)
            if r is not None:
                return r
    return None


def _classify_run(run_path):
    d = json.loads(Path(run_path).read_text(encoding="utf-8"))
    if not d.get("success"):
        return {"success": False, "crs": None, "max_distance": None, "match_count": None}
    meta = (d.get("output") or {}).get("metadata", {})
    return {
        "success": True,
        "crs": _find_key_deep(meta, "target_crs") or _find_key_deep(meta, "crs"),
        "max_distance": _find_key_deep(meta, "max_distance"),
        "match_count": meta.get("output_feature_count"),
    }


rows = []
for i in range(N_RUNS_MITIGATED):
    rows.append(_classify_run(LLM_RUNS_MITIGATED / f"run_{i:02d}.json"))

mitigated_detail = pd.DataFrame(rows)
n_correct_crs = int((mitigated_detail["crs"] == "EPSG:32635").sum())
n_nonzero_match = int(
    (mitigated_detail["success"] & mitigated_detail["match_count"].fillna(0).gt(0)).sum()
)

comparison = {
    "unmitigated": {
        "n_runs": N_RUNS,
        "n_success": int(manifest["success"].sum()),
    },
    "mitigated": {
        "n_runs": N_RUNS_MITIGATED,
        "n_success": n_success_mitigated,
        "n_correct_crs_EPSG_32635": n_correct_crs,
        "n_nonzero_underserved_match": n_nonzero_match,
    },
}
print(json.dumps(comparison, indent=2, ensure_ascii=False))

{
  "unmitigated": {
    "n_runs": 20,
    "n_success": 20
  },
  "mitigated": {
    "n_runs": 20,
    "n_success": 18,
    "n_correct_crs_EPSG_32635": 18,
    "n_nonzero_underserved_match": 18
  }
}
